In [1]:
!!pip install transformers torch

['Collecting transformers',
 '  Downloading transformers-5.6.1-py3-none-any.whl.metadata (33 kB)',
 'Collecting torch',
 '  Downloading torch-2.11.0-cp312-cp312-win_amd64.whl.metadata (29 kB)',
 'Collecting huggingface-hub<2.0,>=1.5.0 (from transformers)',
 '  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)',
 'Requirement already satisfied: numpy>=1.17 in C:\\Users\\Alay\\miniconda3\\envs\\otpp_Analysis\\Lib\\site-packages (from transformers) (2.4.4)',
 'Requirement already satisfied: packaging>=20.0 in C:\\Users\\Alay\\miniconda3\\envs\\otpp_Analysis\\Lib\\site-packages (from transformers) (26.0)',
 'Collecting pyyaml>=5.1 (from transformers)',
 '  Downloading pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)',
 'Collecting regex>=2025.10.22 (from transformers)',
 '  Downloading regex-2026.4.4-cp312-cp312-win_amd64.whl.metadata (41 kB)',
 'Collecting tokenizers<=0.23.0,>=0.22.0 (from transformers)',
 '  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.w

In [4]:
!!pip install tqdm

['Requirement already satisfied: tqdm in C:\\Users\\Alay\\miniconda3\\envs\\otpp_Analysis\\Lib\\site-packages (4.67.3)',
 'Requirement already satisfied: colorama in C:\\Users\\Alay\\miniconda3\\envs\\otpp_Analysis\\Lib\\site-packages (from tqdm) (0.4.6)']

In [2]:
from transformers import pipeline
from tqdm import tqdm
import numpy as np
import pandas as pd
import libsql
import os
from dotenv import load_dotenv
import re
load_dotenv()

try:
    # Testing the native driver
    conn = libsql.connect(database=os.getenv("TURSO_URL"), auth_token=os.getenv("TURSO_TOKEN"))
    cursor = conn.cursor()
except Exception as e:
    print(f"❌ Environment Error: {e}")

In [2]:
finbert = pipeline("text-classification", model="ProsusAI/finbert", top_k=None)

with open("10-Q_2025-10-29_clean.txt", "r") as file:
    
    text = file.read()
    op = finbert(text[:1500])
    print(op[0])
    
    


Loading weights: 100%|██████████| 201/201 [00:00<?, ?it/s]


[{'label': 'neutral', 'score': 0.4241848289966583}, {'label': 'positive', 'score': 0.3591633439064026}, {'label': 'negative', 'score': 0.21665187180042267}]


In [3]:
from transformers import pipeline
import pandas as pd

def score_chunks_with_finbert(cursor, conn):
    print("Loading FinBERT model into memory (this takes a moment)...")
    # Initialize the HuggingFace pipeline specifically built for finance
    finbert = pipeline("text-classification", model="ProsusAI/finbert", top_k=None)
    
    # Pull all unscored chunks from Turso
    cursor.execute("SELECT chunk_id, content FROM sec_mda_risk")
    rows = cursor.fetchall()
    
    if not rows:
        print("All chunks are already scored!")
        return

    print(f"Scoring {len(rows)} chunks...")
    
    updates = []
    for chunk_id, content in rows:
        # FinBERT has a token limit, so we truncate just in case our chunker missed an edge case
        truncated_content = content[:1500] 
        
        raw_output = finbert(truncated_content)
        
        # Handle HF returning either a list of lists or a flat list
        results = raw_output[0] if isinstance(raw_output[0], list) else raw_output
    
        # Initialize defaults
        pos_score = 0.0
        neg_score = 0.0
        neu_score = 0.0
        # Safely extract scores regardless of what order HF returns them in
        for item in results:
            if isinstance(item, dict): # Double-check to prevent the string error
                if item.get('label') == 'positive':
                    pos_score = item.get('score', 0.0)
                elif item.get('label') == 'negative':
                    neg_score = item.get('score', 0.0)
                elif item.get('label') == 'neutral':
                    neu_score = item.get('score', 0.0)
        
        
        
        # Calculate the Net Sentiment Score
        net_sentiment = pos_score - neg_score
        
        updates.append((net_sentiment,neu_score, chunk_id))
        
        
    print("Pushing sentiment scores back to Turso...")
    # Update the database
    cursor.executemany("""
        UPDATE sec_mda_risk
        SET sentiment_score = ?, neutral_score = ? 
        WHERE chunk_id = ?
    """, updates)
    
    conn.commit()
    print("✅ FinBERT scoring complete. Database updated.")



In [48]:
score_chunks_with_finbert(cursor, conn)

Loading FinBERT model into memory (this takes a moment)...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 60555.60it/s]


Scoring 1924 chunks...
Pushing sentiment scores back to Turso...
✅ FinBERT scoring complete. Database updated.


In [16]:
import json

def flatten_transcript_json(transcript_data):
    """Parses the JSON string and extracts only the spoken text."""
    if not transcript_data:
        return ""
        
    # If it's a string, load it into a Python list
    if isinstance(transcript_data, str):
        try:
            transcript_data = json.loads(transcript_data)
        except json.JSONDecodeError:
            # If it fails to parse, it might already be flat text!
            return transcript_data

    if not isinstance(transcript_data, list):
        return str(transcript_data)

    # Extract ONLY the "text" value and join it
    spoken_lines = [item.get("text", "").strip() for item in transcript_data if "text" in item]
    return " ".join(spoken_lines)


def flatten_turso_transcripts(cursor, conn):
    print("Fetching raw JSON transcripts from Turso...")
    
    # Grab all transcripts
    cursor.execute("SELECT transcript_id, content_prepped, content_qa FROM transcripts")
    rows = cursor.fetchall()
    
    if not rows:
        print("⚠️ No transcripts found in the database.")
        return
        
    updates = []
    print(f"Flattening {len(rows)} transcripts...")
    
    for transcript_id, raw_prepped, raw_qa in rows:
        # Flatten both columns
        clean_prepped = flatten_transcript_json(raw_prepped)
        clean_qa = flatten_transcript_json(raw_qa)
        
        updates.append((clean_prepped, clean_qa, transcript_id))
        
    print("Overwriting database with clean, flat text...")
    
    # Push the clean text back into the exact same columns
    cursor.executemany("""
        UPDATE transcripts 
        SET content_prepped = ?, content_qa = ? 
        WHERE transcript_id = ?
    """, updates)
    
    conn.commit()
    print("✅ Transcripts are flattened and ready for FinBERT!")
    print(updates)
# Execute it
flatten_turso_transcripts(cursor, conn)

Fetching raw JSON transcripts from Turso...
Flattening 29 transcripts...
Overwriting database with clean, flat text...
✅ Transcripts are flattened and ready for FinBERT!
[("Greetings and welcome to the Microsoft Fiscal Year 2026 Second Quarter Earnings Conference Call. At this time, all participants are in a listen-only mode. A question and answer session will follow the formal presentation. If anyone should require operator assistance, please press star zero on your telephone keypad. As a reminder, this conference is being recorded. It is now my pleasure to introduce Jonathan Nielsen, Vice President of Investor Relations. Please go ahead. Good afternoon, and thank you for joining us today. On the call with me are Satya Nadella, Chairman and Chief Executive Officer, Amy Hood, Chief Financial Officer, Alice Jolla, Chief Accounting Officer, and Keith Oliver, Corporate Secretary and Deputy General Counsel. On the Microsoft Investor Relations website, you can find our earnings press releas

In [4]:
def semantic_sentence_splitter(text, max_chars=1500):
    """Splits 'wall of text' into chunks at sentence boundaries (~350-400 tokens)."""
    if not text: return []
    # Regex split: period/exclamation/question followed by space
    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks, current_chunk = [], ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) > max_chars and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk += " " + sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

In [5]:


def score_full_transcripts_in_memory(cursor, conn):
    print("Loading FinBERT into memory...")
    finbert = pipeline("text-classification", model="ProsusAI/finbert", top_k=None)
    
    # 1. Grab BOTH text blocks
    cursor.execute("""
        SELECT transcript_id, content_prepped, content_qa 
        FROM transcripts 
    """)
    rows = cursor.fetchall()
    
    if not rows:
        print("No transcripts found!")
        return
        
    print(f"Scoring {len(rows)} transcripts for Prepped vs Q&A signals...")
    updates = []
    
    for transcript_id, content_prepped, content_qa in rows:
        
        # --- Nested Helper Function ---
        def process_text_block(text_block):
            net_sentiments = []
            neutral_scores = []
            
            if not text_block or len(text_block) < 50:
                return net_sentiments, neutral_scores
                
            # Chunk the text using your semantic splitter
            chunks = semantic_sentence_splitter(text_block, max_chars=1200)
            
            for chunk in chunks:
                raw_output = finbert(chunk)
                results = raw_output[0] if isinstance(raw_output[0], list) else raw_output
                
                pos, neg, neu = 0.0, 0.0, 0.0
                for item in results:
                    if isinstance(item, dict):
                        if item.get('label') == 'positive': pos = item.get('score', 0.0)
                        elif item.get('label') == 'negative': neg = item.get('score', 0.0)
                        elif item.get('label') == 'neutral': neu = item.get('score', 0.0)
                        
                # Track both metrics for this chunk
                net_sentiments.append(pos - neg)
                neutral_scores.append(neu)
                
            return net_sentiments, neutral_scores
        # -------------------------------------------------------------------------
        
        # 2. Score the Prepped Remarks
        p_net, p_neu = process_text_block(content_prepped)
        p_sentiment = np.mean(p_net) if p_net else 0.0
        p_dispersion = np.std(p_net) if p_net else 0.0
        p_neutrality = np.mean(p_neu) if p_neu else 0.0
        
        # 3. Score the Q&A Session
        q_net, q_neu = process_text_block(content_qa)
        q_sentiment = np.mean(q_net) if q_net else 0.0
        q_dispersion = np.std(q_net) if q_net else 0.0
        q_neutrality = np.mean(q_neu) if q_neu else 0.0
            
        updates.append((
            p_sentiment, p_dispersion, p_neutrality, 
            q_sentiment, q_dispersion, q_neutrality, 
            transcript_id
        ))
        
    print(f"Pushing {len(updates)} high-resolution scores back to Turso...")
    
    # 4. Overwrite with separated metrics
    cursor.executemany("""
        UPDATE transcripts 
        SET prepped_sentiment = ?, 
            prepped_dispersion = ?, 
            prepped_neutral = ?,
            qa_sentiment = ?, 
            qa_dispersion = ?, 
            qa_neutral = ?
        WHERE transcript_id = ?
    """, updates)
    
    conn.commit()
    print("✅ Phase 2 Complete: Prepped and Q&A metrics successfully scored and separated.")

In [50]:
score_full_transcripts_in_memory(cursor, conn)

Loading FinBERT into memory...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 51315.06it/s]


Scoring 29 transcripts for Prepped vs Q&A signals...
Pushing 29 high-resolution scores back to Turso...
✅ Phase 2 Complete: Prepped and Q&A metrics successfully scored and separated.


In [6]:
# Assume 'conn' is your active Turso database connection
def fetch_raw_data(conn):
    print("Fetching raw data from Turso...")
    data = {
        'market': pd.read_sql("SELECT * FROM market_data", conn),
        'financials': pd.read_sql("SELECT * FROM financial_filings_raw", conn),
        'sec_mda_risk': pd.read_sql("SELECT * FROM sec_mda_risk", conn),
        'transcripts': pd.read_sql("SELECT * FROM transcripts", conn)
    }
    
    # Quick integrity check
    for key, df in data.items():
        print(f"Loaded {key}: {len(df)} rows.")
    return data

raw_data = fetch_raw_data(conn)

Fetching raw data from Turso...


C:\Users\Alay\AppData\Local\Temp\ipykernel_558484\196873989.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  'market': pd.read_sql("SELECT * FROM market_data", conn),
C:\Users\Alay\AppData\Local\Temp\ipykernel_558484\196873989.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  'financials': pd.read_sql("SELECT * FROM financial_filings_raw", conn),
C:\Users\Alay\AppData\Local\Temp\ipykernel_558484\196873989.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  'sec_mda_risk': pd.read_sql("SELECT * FROM sec_mda_risk", conn)

Loaded market: 2088 rows.
Loaded financials: 28 rows.
Loaded sec_mda_risk: 1924 rows.
Loaded transcripts: 29 rows.


C:\Users\Alay\AppData\Local\Temp\ipykernel_558484\196873989.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  'transcripts': pd.read_sql("SELECT * FROM transcripts", conn)


In [7]:
raw_data['financials'].head()

,filing_id,ticker,filing_date,effective_date,fiscal_year,fiscal_period,revenue,net_income,op_cash_flow,capex,cash_eq,total_liability,total_assets,total_equity
0,MSFT_2026_Q2_FILING,MSFT,2026-01-28,2026-01-29,2026,Q2,8.127300e+10,3.845800e+10,3.575800e+10,-2.987600e+10,2.429600e+10,2.744270e+11,6.653020e+11,3.908750e+11
1,MSFT_2026_Q1_FILING,MSFT,2025-10-29,2025-10-30,2026,Q1,7.767300e+10,2.774700e+10,4.505700e+10,-1.939400e+10,2.884900e+10,2.732750e+11,6.363510e+11,3.630760e+11
2,MSFT_2025_Q4_FILING,MSFT,2025-07-30,2025-07-31,2025,Q4,7.644100e+10,2.723300e+10,4.264700e+10,-1.707900e+10,3.024200e+10,2.755240e+11,6.190030e+11,3.434790e+11
3,MSFT_2025_Q3_FILING,MSFT,2025-04-30,2025-05-01,2025,Q3,7.006600e+10,2.582400e+10,3.704400e+10,-1.674500e+10,2.882800e+10,2.407330e+11,5.626240e+11,3.218910e+11
4,MSFT_2025_Q2_FILING,MSFT,2025-01-29,2025-01-30,2025,Q2,6.963200e+10,2.410800e+10,2.229100e+10,-1.580400e+10,1.748200e+10,2.312030e+11,5.338980e+11,3.026950e+11


In [3]:
from src.feature_builder import NLPFeatureEngine, calculate_financial_ratios,\
                                 add_temporal_signals, build_market_and_vol_features


In [7]:
import pandas as pd
from pandas.tseries.offsets import BDay

def run_pipeline():
    print("--- Starting Final DataFrame Pipeline ---")
    if not conn: return None
    
    raw = fetch_raw_data(conn)
    nlp_pipe = NLPFeatureEngine()
    
    # ==========================================
    # 1. ENGINEER FEATURES
    # ==========================================
    print("Engineering fundamental ratios...")
    df_fundamentals = calculate_financial_ratios(raw['financials'])
    
    print("Engineering NLP features...")
    # Get the raw tables
    df_sec_raw = raw['sec_mda_risk']
    df_trans_raw = raw['transcripts']
    
    # Apply the NLP Engine to get the actual mathematical features
    df_sec_feat = nlp_pipe.aggregate_sec_chunks(df_sec_raw)
    
    # We only need the numeric columns and dates for the merge, not the raw text
    df_trans_feat = df_trans_raw[['event_timestamp',
        'effective_date', 'prepped_sentiment', 'prepped_dispersion', 'prepped_neutral',
        'qa_sentiment', 'qa_dispersion', 'qa_neutral'
    ]].copy()

    # ==========================================
    # 2. ALIGN TIMELINES (CRITICAL STEP)
    # ==========================================
    # Force everything to pure Datetime objects before joining
    df_market = raw['market'].copy()
    df_market['trading_date'] = pd.to_datetime(df_market['trading_date'])
    df_fundamentals['effective_date'] = pd.to_datetime(df_fundamentals['effective_date'])
    df_trans_feat['effective_date'] = pd.to_datetime(df_trans_feat['effective_date'])
    
    # ==========================================
    # 3. BUILD THE GOLDEN DATAFRAME
    # ==========================================
    print("Merging layers into Final Timeline...")
    df_final = df_market.copy().sort_values('trading_date')
    
    # Left join all features onto the daily trading timeline using EFFECTIVE_DATE
    df_final = df_final.merge(df_fundamentals, left_on='trading_date', right_on='effective_date', how='left')
    df_final = df_final.merge(df_sec_feat, left_on='trading_date', right_on='effective_date', how='left', suffixes=('', '_sec'))
    df_final = df_final.merge(df_trans_feat, left_on='trading_date', right_on='effective_date', how='left', suffixes=('', '_trans'))

    
    df_final = add_temporal_signals(df_final)
    # Drop all redundant date columns that the merges created
    cols_to_drop = [c for c in df_final.columns if 'effective_date' in c or 'filing_date' in c or 'event_timestamp' in c ]
    df_final.drop(columns=cols_to_drop, errors='ignore', inplace=True)

    # ==========================================
    # 4. FORWARD-FILL STATIONARY METRICS
    # ==========================================
    print("Applying Forward-Fill for stationary metrics...")
    
    # Updated to match the exact output of your new Feature Engine
    cols_to_ffill = [
        # Fundamentals
        'roa', 'debt_to_asset', 'fcf_margin', 'ni_growth_qoq',
        # SEC NLP
        'sec_sentiment', 'sec_neutral', 'sec_dispersion',
        # Transcript NLP
        'prepped_sentiment', 'prepped_neutral', 'prepped_dispersion',
        'qa_sentiment', 'qa_neutral', 'qa_dispersion'
    ]
    
    # Only ffill columns that actually successfully joined
    existing_cols = [c for c in cols_to_ffill if c in df_final.columns]
    df_final[existing_cols] = df_final[existing_cols].ffill()
    
    
    # Build Volatility & Targets (from feature_factory)
    df_final = build_market_and_vol_features(df_final) # Uncomment if you have this function ready
    
    df_final = df_final.fillna(0)
    
    print(f"✅ df_final successfully built! Shape: {df_final.shape}")
    return df_final

In [8]:
df = run_pipeline()

--- Starting Final DataFrame Pipeline ---
Fetching raw data from Turso...


C:\Users\Alay\AppData\Local\Temp\ipykernel_558484\196873989.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  'market': pd.read_sql("SELECT * FROM market_data", conn),
C:\Users\Alay\AppData\Local\Temp\ipykernel_558484\196873989.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  'financials': pd.read_sql("SELECT * FROM financial_filings_raw", conn),
C:\Users\Alay\AppData\Local\Temp\ipykernel_558484\196873989.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  'sec_mda_risk': pd.read_sql("SELECT * FROM sec_mda_risk", conn)

Loaded market: 2088 rows.
Loaded financials: 28 rows.
Loaded sec_mda_risk: 1924 rows.
Loaded transcripts: 29 rows.
Loading ProsusAI/finbert into memory...


Loading weights: 100%|██████████| 201/201 [00:00<?, ?it/s]


Engineering fundamental ratios...
Engineering NLP features...
Aggregating 1924 SEC chunks...
Merging layers into Final Timeline...
⏳ Calculating Information Decay (Temporal Signals)...
Applying Forward-Fill for stationary metrics...
📈 Engineering Market, Macro, Sector, and Volatility Signals...
✅ df_final successfully built! Shape: (2088, 41)


,trading_date,msft_close,msft_volume,msft_return,vix_close,tnx_yield,msft_open,msft_high,msft_low,qqq_close,...,vix_5d_trend,yield_10y_level,yield_10y_delta_5d,vol_rolling_21d,qqq_vol_21d,vol_garman_klass,vol_surge,amihud_ratio,dist_from_ma200,target_vol_21d
0,2018-01-03,79.237419,26061400,0.000000,9.15,2.447,78.971305,79.384243,78.888722,151.681824,...,0.0,2.447,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0
1,2018-01-04,79.934799,21912000,0.008801,9.22,2.453,79.457627,80.439499,79.439278,151.947189,...,0.0,2.453,0.0,0.0,0.0,0.0,0.0,5.024829e-06,0.0,0.0
2,2018-01-05,80.925850,23407100,0.012398,9.22,2.476,80.439507,81.127730,80.228449,153.473236,...,0.0,2.476,0.0,0.0,0.0,0.0,0.0,6.545233e-06,0.0,0.0
3,2018-01-08,81.008446,22113000,0.001021,9.52,2.480,80.935034,81.283738,80.384457,154.070343,...,0.0,2.480,0.0,0.0,0.0,0.0,0.0,5.697612e-07,0.0,0.0
4,2018-01-09,80.953384,19484300,-0.000680,10.08,2.546,81.347966,81.421378,80.623037,154.079834,...,0.0,2.546,0.0,0.0,0.0,0.0,0.0,4.309200e-07,0.0,0.0


In [62]:
df_fin = build_financial_ratios(raw_data['financials'])

In [71]:
df_fin.head()

,filing_id,ticker,filing_date,effective_date,fiscal_year,fiscal_period,revenue,net_income,op_cash_flow,capex,cash_eq,total_liability,total_assets,total_equity,roa,debt_to_asset,fcf_margin,ni_growth_qoq
27,MSFT_2019_Q2_FILING,MSFT,2019-01-30,2019-01-31,2019,Q2,3.247100e+10,8.420000e+09,8.900000e+09,-3.707000e+09,6.638000e+09,1.667310e+11,2.588590e+11,9.212800e+10,0.032527,0.644100,0.159927,NaN
26,MSFT_2019_Q3_FILING,MSFT,2019-04-24,2019-04-25,2019,Q3,3.057100e+10,8.809000e+09,1.352000e+10,-2.565000e+09,1.121200e+10,1.684170e+11,2.632810e+11,9.486400e+10,0.033459,0.639685,0.358346,0.046200
25,MSFT_2020_Q1_FILING,MSFT,2019-10-23,2019-10-24,2020,Q1,3.305500e+10,1.067800e+10,1.381800e+10,-3.385000e+09,1.311700e+10,1.728940e+11,2.789550e+11,1.060610e+11,0.038279,0.619792,0.315625,0.212169
24,MSFT_2020_Q2_FILING,MSFT,2020-01-29,2020-01-30,2020,Q2,3.690600e+10,1.164900e+10,1.068000e+10,-3.545000e+09,8.864000e+09,1.726850e+11,2.827940e+11,1.101090e+11,0.041193,0.610639,0.193329,0.090935
23,MSFT_2020_Q3_FILING,MSFT,2020-04-29,2020-04-30,2020,Q3,3.502100e+10,1.075200e+10,1.750400e+10,-3.767000e+09,1.171000e+10,1.709480e+11,2.854490e+11,1.145010e+11,0.037667,0.598874,0.392250,-0.077002


In [65]:
df_market = build_volatility_signals(raw_data['market'])

0            NaN
1            NaN
2            NaN
3            NaN
4            NaN
          ...   
1850    0.301913
1851    0.304093
1852    0.295406
1853    0.300055
1854    0.298412
Name: vol_rolling_21d, Length: 1855, dtype: float64